# EIT: Extreme-ultraviolet Imaging Telescope — Implementation / 구현

**Paper**: Delaboudinière, J.-P., et al. (1995). "EIT: Extreme-ultraviolet Imaging Telescope for the SOHO Mission." *Solar Physics*, 162, 291–312. [DOI: 10.1007/BF00733432]

This notebook implements the key optical, detector, and data-processing concepts of the EIT instrument described in the paper. We simulate multilayer reflectivity, model the optical design, compute the CCD signal chain, derive temperature response functions, analyze data compression, and compare EIT with its successors.

이 노트북은 논문에 기술된 EIT 기기의 핵심 광학, 검출기, 데이터 처리 개념을 구현합니다. 다층 코팅 반사율 시뮬레이션, 광학 설계 모델링, CCD 신호 체계 계산, 온도 응답 함수 도출, 데이터 압축 분석, 그리고 EIT와 후속 기기들의 비교를 수행합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.fft import dctn, idctn

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# EIT band definitions used throughout the notebook.
EIT_BANDS = {
    '171': {'center': 171.0, 'color': '#1f77b4', 'ion': 'Fe IX/X'},
    '195': {'center': 195.0, 'color': '#2ca02c', 'ion': 'Fe XII'},
    '284': {'center': 284.0, 'color': '#ff7f0e', 'ion': 'Fe XV'},
    '304': {'center': 304.0, 'color': '#d62728', 'ion': 'He II'},
}

## Part 1: Multilayer Reflectivity / 다층 코팅 반사율

EIT uses Mo/Si multilayer-coated mirrors to achieve narrow-band EUV reflectivity. Each mirror quadrant has a different bilayer period $d$ tuned to a specific wavelength via the Bragg condition:

$$\lambda = 2d\sin\theta$$

EIT는 Mo/Si 다층 코팅 거울을 사용하여 협대역 EUV 반사율을 달성합니다. 각 거울 사분면은 Bragg 조건에 의해 특정 파장에 맞춰진 서로 다른 이중층 주기 $d$를 가집니다.

We model reflectivity using a simplified transfer matrix approach with absorption. Mo acts as the high-Z absorber and Si as the low-Z spacer. The bilayer ratio is $\Gamma = d_{Mo}/d \approx 0.4$ (Mo occupies 40% of each period).

흡수를 포함한 간소화된 전달 행렬 접근법으로 반사율을 모델링합니다. Mo는 고-Z 흡수체, Si는 저-Z 스페이서 역할을 합니다. 이중층 비율은 $\Gamma = d_{Mo}/d \approx 0.4$입니다.

**Parameters from the paper (Table I, Fig. 5):**
- Bilayer periods: 171 A -> d ~ 88 A, 195 A -> d ~ 100 A, 284 A -> d ~ 145 A, 304 A -> d ~ 155 A
- Number of bilayers: N = 20-40
- Peak two-bounce reflectivity: ~3-7%

In [ ]:
def multilayer_reflectivity(wavelength, d_bilayer, n_bilayers, gamma=0.4,
                             theta=np.pi / 2):
    """Compute Mo/Si multilayer reflectivity using a simplified Bragg model.

    Uses the kinematic (Born) approximation with absorption damping:
        R(lambda) = R_interface^2 * sin^2(N * phi/2) / sin^2(phi/2) * exp(-sigma)

    where phi is the phase shift per bilayer and the Debye-Waller-like
    factor accounts for interfacial roughness and absorption.

    Args:
        wavelength: Array of wavelengths in Angstroms.
        d_bilayer: Bilayer period in Angstroms.
        n_bilayers: Number of Mo/Si bilayer pairs.
        gamma: Fraction of bilayer occupied by Mo (absorber).
        theta: Grazing angle in radians (pi/2 for normal incidence).

    Returns:
        Single-bounce reflectivity array.
    """
    # Optical constants (simplified, energy-averaged for EUV).
    # delta and beta are the real and imaginary parts of (1 - n).
    delta_mo = 0.07
    beta_mo = 0.006
    delta_si = 0.02
    beta_si = 0.001

    # Fresnel reflectivity amplitude at Mo/Si interface (normal incidence).
    delta_n = np.sqrt((delta_mo - delta_si)**2 + (beta_mo - beta_si)**2)
    r_interface = delta_n / 2.0

    # Phase shift per bilayer (Bragg condition).
    phi = 2.0 * np.pi * 2.0 * d_bilayer * np.sin(theta) / wavelength

    # Multilayer interference (kinematic approximation).
    half_phi = phi / 2.0
    numerator = np.sin(n_bilayers * half_phi) ** 2
    denominator = np.sin(half_phi) ** 2
    # Avoid division by zero at exact Bragg condition.
    denominator = np.where(denominator < 1e-15, 1e-15, denominator)

    # Absorption damping factor: reduces reflectivity away from peak.
    # Effective absorption length scales with Mo fraction.
    mu_eff = 4.0 * np.pi * (gamma * beta_mo + (1 - gamma) * beta_si)
    absorption = np.exp(-mu_eff * n_bilayers * d_bilayer / wavelength)

    # Interfacial roughness damping (Debye-Waller, sigma ~ 5 Angstroms).
    sigma_roughness = 5.0  # Angstroms
    dw_factor = np.exp(-2.0 * (2.0 * np.pi * sigma_roughness / wavelength)**2)

    reflectivity = (r_interface**2 * numerator / denominator
                    * absorption * dw_factor)

    # Normalize peak to realistic values from paper Fig. 5.
    return reflectivity


# --- EIT quadrant parameters ---
quadrant_params = {
    '171': {'d': 88.0, 'N': 32, 'peak_R2': 0.035},
    '195': {'d': 100.0, 'N': 28, 'peak_R2': 0.070},
    '284': {'d': 145.0, 'N': 22, 'peak_R2': 0.042},
    '304': {'d': 155.0, 'N': 20, 'peak_R2': 0.030},
}

# Compute and normalize reflectivity curves.
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Mo/Si Multilayer Reflectivity — EIT Quadrants (cf. Fig. 5)',
             fontsize=14, fontweight='bold')

for ax, (band, params) in zip(axes.flat, quadrant_params.items()):
    info = EIT_BANDS[band]
    wl = np.linspace(params['d'] * 1.5, params['d'] * 2.5, 1000)
    r_single = multilayer_reflectivity(wl, params['d'], params['N'])

    # Normalize so that peak two-bounce reflectivity matches paper values.
    peak_single = np.sqrt(params['peak_R2'])
    r_single = r_single / r_single.max() * peak_single
    r_two_bounce = r_single ** 2

    ax.plot(wl, r_single * 100, color=info['color'], linewidth=1.5,
            label='Single surface')
    ax.plot(wl, r_two_bounce * 100, color=info['color'], linewidth=2.5,
            linestyle='--', label='Two-bounce')
    ax.axvline(info['center'], color='gray', linestyle=':', alpha=0.6)
    ax.set_title(f"{band} A  ({info['ion']})", fontsize=12)
    ax.set_xlabel('Wavelength (A)')
    ax.set_ylabel('Reflectivity (%)')
    ax.legend(fontsize=9)
    ax.set_xlim(info['center'] - 30, info['center'] + 30)
    peak_pct = params['peak_R2'] * 100
    ax.annotate(f'Peak R2 = {peak_pct:.1f}%',
                xy=(info['center'], peak_pct),
                xytext=(info['center'] + 10, peak_pct * 0.8),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

# Print summary table.
print("\n--- Multilayer Parameters Summary ---")
print(f"{'Band':>6s} {'d (A)':>8s} {'N':>5s} {'Peak R1 (%)':>12s} {'Peak R2 (%)':>12s}")
for band, params in quadrant_params.items():
    r1 = np.sqrt(params['peak_R2']) * 100
    r2 = params['peak_R2'] * 100
    print(f"{band:>6s} {params['d']:>8.0f} {params['N']:>5d} {r1:>12.1f} {r2:>12.1f}")

## Part 2: EIT Optical Design / EIT 광학 설계

EIT is a Ritchey-Chretien telescope (two hyperbolic mirrors) with a 12 cm primary aperture and 165.2 cm effective focal length. The unique feature is the **4-quadrant mirror design**: each quadrant of both the primary and secondary mirrors is coated with a different multilayer, and a rotating mask selects one quadrant at a time.

EIT는 12 cm 주경 구경과 165.2 cm 유효 초점 거리를 가진 Ritchey-Chretien 망원경(두 개의 쌍곡면 거울)입니다. 독특한 특징은 **4사분면 거울 설계**로, 주경과 부경의 각 사분면에 서로 다른 다층 코팅을 하고, 회전 마스크가 한 번에 하나의 사분면을 선택합니다.

**Key optical parameters (Table I):**
- Primary mirror diameter: 12 cm
- Secondary mirror diameter: ~4 cm
- Effective focal length: 1652 mm
- Pixel size: 21 um -> 2.62"/pixel
- Detector: 1024 x 1024 CCD
- FOV: 45' x 45' (covers full Sun plus ~0.5 R_sun beyond limb)

In [ ]:
# =====================================================================
# Part 2a: Ray trace diagram of Ritchey-Chretien telescope
# =====================================================================
fig, ax = plt.subplots(figsize=(14, 5))

# Mirror positions (schematic, not to scale for clarity).
primary_z = 0.0      # Primary mirror at z = 0
secondary_z = 50.0   # Secondary mirror
focal_z = 165.2      # Focal plane

# Primary mirror (concave parabola-like curve).
y_primary = np.linspace(-6, 6, 100)
z_primary = primary_z + 0.02 * y_primary**2  # Slight curvature
ax.plot(z_primary, y_primary, 'b-', linewidth=3, label='Primary (12 cm)')

# Secondary mirror (convex).
y_secondary = np.linspace(-2, 2, 50)
z_secondary = secondary_z - 0.05 * y_secondary**2
ax.plot(z_secondary, y_secondary, 'r-', linewidth=3, label='Secondary (~4 cm)')

# Focal plane.
ax.plot([focal_z, focal_z], [-3, 3], 'k-', linewidth=2, label='Focal plane (CCD)')

# Ray traces (3 parallel rays from the left).
for y_in in [-4.0, 0.0, 4.0]:
    # Incoming ray -> primary.
    ax.annotate('', xy=(primary_z + 0.02 * y_in**2, y_in),
                xytext=(-20, y_in),
                arrowprops=dict(arrowstyle='->', color='gold', lw=1.2))
    # Primary -> secondary (converging).
    y_sec = y_in * 0.33  # Magnification
    ax.plot([primary_z + 0.02 * y_in**2, secondary_z - 0.05 * y_sec**2],
            [y_in, y_sec], 'gold', linewidth=1.2)
    # Secondary -> focal plane (diverging through hole in primary).
    ax.plot([secondary_z - 0.05 * y_sec**2, focal_z],
            [y_sec, 0.0], 'gold', linewidth=1.2, alpha=0.7)

# Quadrant mask (schematic).
mask_z = -5
ax.add_patch(plt.Circle((mask_z, 0), 3, fill=False, edgecolor='purple',
                         linewidth=2, linestyle='--'))
ax.text(mask_z, 4.5, 'Rotating\nmask', ha='center', fontsize=9, color='purple')

# Draw quadrant lines on primary.
ax.plot([primary_z, primary_z], [-6, 6], 'b-', alpha=0.3)
ax.plot([primary_z - 0.5, primary_z + 1.5], [0, 0], 'b--', alpha=0.3)

# Labels.
ax.text(primary_z + 2, -7.5, 'Primary\n(12 cm)', ha='center', fontsize=10)
ax.text(secondary_z, -3.5, 'Secondary', ha='center', fontsize=10)
ax.text(focal_z, -4.5, 'CCD\n(1024x1024)', ha='center', fontsize=10)
ax.text(80, 5, f'f_eff = {focal_z} cm', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Quadrant inset: show 4 quadrants of primary mirror.
ax_inset = fig.add_axes([0.12, 0.55, 0.15, 0.35])
theta_q = np.linspace(0, 2 * np.pi, 200)
ax_inset.plot(6 * np.cos(theta_q), 6 * np.sin(theta_q), 'b-', linewidth=2)
ax_inset.plot(2 * np.cos(theta_q), 2 * np.sin(theta_q), 'k--', linewidth=1)
ax_inset.axhline(0, color='k', linewidth=0.5)
ax_inset.axvline(0, color='k', linewidth=0.5)
colors_q = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
labels_q = ['171', '195', '284', '304']
positions_q = [(3, 3), (-3, 3), (-3, -3), (3, -3)]
for (x, y), c, lbl in zip(positions_q, colors_q, labels_q):
    ax_inset.text(x, y, f'{lbl} A', ha='center', va='center',
                  fontsize=8, fontweight='bold', color=c)
ax_inset.set_title('Mirror quadrants', fontsize=8)
ax_inset.set_xlim(-8, 8)
ax_inset.set_ylim(-8, 8)
ax_inset.set_aspect('equal')
ax_inset.axis('off')

ax.set_xlim(-25, 175)
ax.set_ylim(-10, 10)
ax.set_xlabel('Optical axis (cm)')
ax.set_ylabel('Height (cm)')
ax.set_title('EIT Ritchey-Chretien Telescope — Schematic Ray Trace',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('auto')
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Part 2b: FOV calculation and solar disk overlay
# =====================================================================
# Plate scale calculation.
pixel_size_um = 21.0         # um
focal_length_mm = 1652.0     # mm
pixel_size_mm = pixel_size_um / 1000.0

# Angular size per pixel (radians -> arcseconds).
pixel_rad = pixel_size_mm / focal_length_mm
pixel_arcsec = pixel_rad * 206265.0  # 1 rad = 206265 arcsec
print(f"Plate scale: {pixel_arcsec:.2f} arcsec/pixel")

fov_arcsec = 1024 * pixel_arcsec
fov_arcmin = fov_arcsec / 60.0
print(f"FOV: {fov_arcmin:.1f}' x {fov_arcmin:.1f}'")
print(f"FOV: {fov_arcsec:.0f}\" x {fov_arcsec:.0f}\"")

# Solar disk angular radius.
solar_radius_arcsec = 960.0   # ~16' = 960"
solar_diameter_arcmin = 2 * solar_radius_arcsec / 60.0
print(f"Solar diameter: {solar_diameter_arcmin:.0f}' (angular)")
print(f"EIT reaches: {fov_arcsec / 2 / solar_radius_arcsec:.2f} R_sun from center")

# --- Plot FOV overlaid on solar disk ---
fig, ax = plt.subplots(figsize=(8, 8))

# Solar disk.
theta_circle = np.linspace(0, 2 * np.pi, 300)
ax.plot(solar_radius_arcsec * np.cos(theta_circle),
        solar_radius_arcsec * np.sin(theta_circle),
        'orange', linewidth=2.5, label='Solar limb (1.0 R_sun)')
ax.fill(solar_radius_arcsec * np.cos(theta_circle),
        solar_radius_arcsec * np.sin(theta_circle),
        color='gold', alpha=0.15)

# 1.5 R_sun ring.
r15 = 1.5 * solar_radius_arcsec
ax.plot(r15 * np.cos(theta_circle), r15 * np.sin(theta_circle),
        'orange', linewidth=1, linestyle=':', alpha=0.5, label='1.5 R_sun')

# EIT FOV (square).
half_fov = fov_arcsec / 2
rect = plt.Rectangle((-half_fov, -half_fov), fov_arcsec, fov_arcsec,
                      fill=False, edgecolor='blue', linewidth=2.5,
                      linestyle='--', label=f"EIT FOV ({fov_arcmin:.0f}')")
ax.add_patch(rect)

ax.set_xlim(-1600, 1600)
ax.set_ylim(-1600, 1600)
ax.set_aspect('equal')
ax.set_xlabel('E-W (arcsec)')
ax.set_ylabel('N-S (arcsec)')
ax.set_title('EIT Field of View vs Solar Disk', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.3)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Part 2c: Point Spread Function (PSF)
# =====================================================================
# The paper reports a PSF narrower than the 21 um pixel.
# The effective resolution is ~2.6" (pixel-limited).
# Model the optical PSF as a 2D Gaussian.

pixel_size_arcsec = pixel_arcsec  # ~2.62"
psf_fwhm_arcsec = 1.8  # Optical PSF FWHM (sub-pixel)
psf_sigma_arcsec = psf_fwhm_arcsec / (2.0 * np.sqrt(2.0 * np.log(2.0)))

# Create 2D grid in arcseconds.
x_arcsec = np.linspace(-8, 8, 500)
y_arcsec = np.linspace(-8, 8, 500)
X, Y = np.meshgrid(x_arcsec, y_arcsec)
R2 = X**2 + Y**2

# Optical PSF (Gaussian).
psf_optical = np.exp(-R2 / (2.0 * psf_sigma_arcsec**2))
psf_optical /= psf_optical.sum()

# Pixel response (top-hat convolution, 2.62" square).
pixel_response = np.where(
    (np.abs(X) < pixel_size_arcsec / 2) & (np.abs(Y) < pixel_size_arcsec / 2),
    1.0, 0.0)
pixel_response /= pixel_response.sum()

# Effective PSF = optical PSF convolved with pixel (approximate as sum).
# For visualization, show profiles through center.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D PSF image.
ax = axes[0]
im = ax.imshow(psf_optical / psf_optical.max(), extent=[-8, 8, -8, 8],
               cmap='inferno', origin='lower')
# Overlay pixel grid.
for offset in np.arange(-8, 9, pixel_size_arcsec):
    ax.axhline(offset, color='white', linewidth=0.3, alpha=0.4)
    ax.axvline(offset, color='white', linewidth=0.3, alpha=0.4)
ax.set_xlabel('arcsec')
ax.set_ylabel('arcsec')
ax.set_title('EIT Optical PSF (2D Gaussian)', fontsize=12)
plt.colorbar(im, ax=ax, label='Normalized intensity')

# 1D profiles.
ax = axes[1]
center_idx = len(y_arcsec) // 2
profile_optical = psf_optical[center_idx, :] / psf_optical[center_idx, :].max()
profile_pixel = pixel_response[center_idx, :] / max(pixel_response[center_idx, :].max(), 1e-15)

ax.plot(x_arcsec, profile_optical, 'r-', linewidth=2,
        label=f'Optical PSF (FWHM={psf_fwhm_arcsec:.1f}")')
ax.plot(x_arcsec, profile_pixel, 'b--', linewidth=2,
        label=f'Pixel ({pixel_size_arcsec:.1f}" square)')

# Effective PSF (convolution of the two profiles, approximate).
from scipy.ndimage import uniform_filter1d
effective_profile = uniform_filter1d(profile_optical,
                                      size=int(pixel_size_arcsec / (x_arcsec[1] - x_arcsec[0])))
effective_profile /= effective_profile.max()
ax.plot(x_arcsec, effective_profile, 'k-', linewidth=2.5,
        label='Effective PSF')

ax.set_xlabel('Position (arcsec)')
ax.set_ylabel('Normalized intensity')
ax.set_title('PSF Cross-section (cf. Fig. 3/4)', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(-6, 6)

plt.suptitle('EIT Point Spread Function', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Compute effective FWHM.
half_max_indices = np.where(effective_profile >= 0.5)[0]
if len(half_max_indices) > 0:
    eff_fwhm = x_arcsec[half_max_indices[-1]] - x_arcsec[half_max_indices[0]]
    print(f"Effective resolution (FWHM): {eff_fwhm:.1f} arcsec")
    print(f"Pixel size: {pixel_size_arcsec:.2f} arcsec")
    print(f"Optical PSF FWHM: {psf_fwhm_arcsec:.1f} arcsec")

## Part 3: CCD Quantum Efficiency and Signal Chain / CCD 양자 효율 및 신호 체계

The EIT CCD is a thinned, back-illuminated 1024x1024 pixel device with a lumogen phosphor coating for enhanced EUV sensitivity. The complete signal chain converts incident EUV photons to digital numbers (DN).

EIT CCD는 EUV 감도를 높이기 위해 lumogen 형광체 코팅이 된 박형 후면 조사 1024x1024 pixel 소자입니다. 전체 신호 체계는 입사 EUV 광자를 디지털 숫자(DN)로 변환합니다.

**Signal chain**: photon -> CCD absorption -> electron-hole pairs -> analog signal -> 14-bit ADC -> DN
- Electron-hole pair creation energy in Si: 3.65 eV/pair
- Gain: 18 electrons/DN (Table III)
- QE data from Table III: 0.36 (171 A), 0.34 (195 A), 0.29 (284 A), 0.27 (304 A)

In [ ]:
# =====================================================================
# Part 3a: CCD Quantum Efficiency model
# =====================================================================
# QE data from Table III.
qe_data = {
    'wavelength_A': np.array([171, 195, 284, 304, 550 * 10, 800 * 10]),
    'qe': np.array([0.36, 0.34, 0.29, 0.27, 0.20, 0.02])
}
# Note: 550 and 800 nm converted to Angstroms for consistent units.

# Simplified QE model: QE depends on photon absorption in the active
# layer and dead layer absorption.
# QE(lambda) = (1 - R) * (1 - exp(-alpha * d_active)) * exp(-alpha_dead * d_dead)
# For EUV, absorption is very high, so QE ~ transmission through dead layer.

def qe_model(wavelength_A, qe_max, dead_layer_A, decay_rate):
    """Simplified CCD quantum efficiency model.

    Args:
        wavelength_A: Wavelength in Angstroms.
        qe_max: Maximum QE at optimal wavelength.
        dead_layer_A: Dead layer thickness in Angstroms.
        decay_rate: Rate of QE falloff at long wavelengths.

    Returns:
        Quantum efficiency array.
    """
    # EUV photon energy in eV.
    energy_ev = 12400.0 / wavelength_A

    # Absorption probability in active silicon (high for EUV, low for visible).
    # Absorption length ~ lambda/4pi*beta, very short for EUV.
    absorption_prob = 1.0 - np.exp(-energy_ev / 5.0)

    # Dead layer transmission (thin oxide/lumogen layer attenuates).
    dead_layer_trans = np.exp(-dead_layer_A / (wavelength_A + 50.0))

    # Long-wavelength cutoff.
    long_wl_factor = np.exp(-decay_rate * wavelength_A / 1000.0)

    return qe_max * absorption_prob * dead_layer_trans * long_wl_factor


# Fit model to data.
wl_data = qe_data['wavelength_A']
qe_measured = qe_data['qe']

popt, pcov = curve_fit(qe_model, wl_data, qe_measured,
                       p0=[0.5, 100.0, 0.3], maxfev=10000)
print(f"Fit parameters: QE_max={popt[0]:.3f}, dead_layer={popt[1]:.0f} A, "
      f"decay_rate={popt[2]:.4f}")

# Plot QE curve.
fig, ax = plt.subplots(figsize=(10, 6))

wl_plot = np.logspace(np.log10(100), np.log10(10000), 500)
qe_fit = qe_model(wl_plot, *popt)

ax.semilogx(wl_plot, qe_fit * 100, 'b-', linewidth=2, label='QE model fit')
ax.scatter(wl_data, qe_measured * 100, color='red', s=100, zorder=5,
           label='Table III data')

# Mark EIT bands.
for band, info in EIT_BANDS.items():
    wl = info['center']
    qe_val = qe_model(wl, *popt)
    ax.axvline(wl, color=info['color'], linestyle='--', alpha=0.5)
    ax.text(wl, qe_val * 100 + 2, f'{band} A', ha='center', fontsize=9,
            color=info['color'])

ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Quantum Efficiency (%)')
ax.set_title('EIT CCD Quantum Efficiency (cf. Table III)', fontsize=13,
             fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(100, 10000)
ax.set_ylim(0, 50)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Part 3b & 3c: Electron-hole pair generation and DN conversion
# =====================================================================
SI_EH_ENERGY = 3.65  # eV per electron-hole pair in silicon
GAIN_E_PER_DN = 18.0  # electrons per DN (Table III)

print("=" * 65)
print("Electron-Hole Pair Generation and DN Conversion per EIT Band")
print("=" * 65)
print(f"{'Band':>6s} {'Wl (A)':>8s} {'E (eV)':>8s} {'n_eh':>8s} "
      f"{'QE':>6s} {'e-/photon':>10s} {'DN/photon':>10s}")
print("-" * 65)

band_signal = {}
qe_table = {'171': 0.36, '195': 0.34, '284': 0.29, '304': 0.27}

for band, info in EIT_BANDS.items():
    wl = info['center']
    energy_ev = 12400.0 / wl
    n_eh = energy_ev / SI_EH_ENERGY
    qe = qe_table[band]
    electrons_per_photon = n_eh * qe
    dn_per_photon = electrons_per_photon / GAIN_E_PER_DN

    band_signal[band] = {
        'energy_ev': energy_ev, 'n_eh': n_eh, 'qe': qe,
        'e_per_photon': electrons_per_photon, 'dn_per_photon': dn_per_photon
    }

    print(f"{band:>6s} {wl:>8.0f} {energy_ev:>8.1f} {n_eh:>8.1f} "
          f"{qe:>6.2f} {electrons_per_photon:>10.1f} {dn_per_photon:>10.2f}")

print("\nNote: DN/photon = (E_photon / 3.65 eV) * QE / (18 e-/DN)")

In [ ]:
# =====================================================================
# Part 3d: Signal prediction for different solar features (cf. Table V)
# =====================================================================
# Effective area: A_eff = A_geom * R_two_bounce * filter_trans * QE
# A_geom for one quadrant ~ (pi/4) * (12/2)^2 / 4 = ~28 cm^2
A_geom_quadrant = np.pi * (12.0 / 2)**2 / 4.0  # cm^2 per quadrant

# Pixel solid angle (steradians).
pixel_sr = (pixel_arcsec / 206265.0)**2

# Approximate effective areas from paper context (cm^2).
eff_area = {'171': 0.15, '195': 0.30, '284': 0.10, '304': 0.08}

# Typical radiances (photons/cm^2/s/sr/A) for solar features.
features = {
    'Coronal Hole':  {'171': 5e8,  '195': 3e8,  '284': 1e8,  '304': 2e9},
    'Quiet Sun':     {'171': 2e9,  '195': 1e9,  '284': 5e8,  '304': 5e9},
    'Active Region': {'171': 1e10, '195': 8e9,  '284': 3e9,  '304': 2e10},
    'Flare (M1)':    {'171': 5e10, '195': 5e10, '284': 2e10, '304': 5e10},
}

# Bandpass widths (FWHM, Angstroms) from multilayer reflectivity.
bandpass = {'171': 8.0, '195': 10.0, '284': 12.0, '304': 15.0}

# Typical exposure times (seconds).
exposure = {'171': 15.0, '195': 15.0, '284': 30.0, '304': 15.0}

print("=" * 75)
print("Expected Signal (DN/pixel) for Solar Features (cf. Table V concept)")
print("=" * 75)
print(f"{'Feature':<18s}", end="")
for band in EIT_BANDS:
    print(f" {band:>10s} A", end="")
print()
print("-" * 75)

for feat_name, radiances in features.items():
    print(f"{feat_name:<18s}", end="")
    for band in EIT_BANDS:
        # Signal = radiance * bandpass * eff_area * pixel_sr * exposure * dn_per_photon
        sig = (radiances[band] * bandpass[band] * eff_area[band]
               * pixel_sr * exposure[band] * band_signal[band]['dn_per_photon'])
        print(f" {sig:>10.0f} DN", end="")
    print()

print(f"\n(Exposure times: ", end="")
for band in EIT_BANDS:
    print(f"{band}A={exposure[band]:.0f}s  ", end="")
print(")")
print(f"14-bit ADC saturation: {2**14} DN")

## Part 4: Temperature Response Functions / 온도 응답 함수

The temperature response function $G(T)$ describes how much signal (DN/s) EIT produces from plasma at temperature $T$. It is the key function that enables temperature diagnostics from EIT images.

온도 응답 함수 $G(T)$는 온도 $T$의 플라즈마로부터 EIT가 생성하는 신호(DN/s)를 기술합니다. EIT 영상으로부터 온도 진단을 가능하게 하는 핵심 함수입니다.

For each band, $G(T)$ is the product of:
- **Ion fraction**: fraction of the element in the relevant ionization state at temperature $T$
- **Contribution function**: emissivity of that ion
- **Effective area**: mirror reflectivity x filter transmission x CCD QE

We use Gaussian approximations for the ion fractions, centered at the formation temperatures of the dominant ions (cf. Fig. 9).

각 밴드에 대해 $G(T)$는 이온 분율, 기여 함수, 유효 면적의 곱입니다. 지배적 이온의 형성 온도에 중심을 둔 Gaussian 근사를 사용합니다.

In [ ]:
# =====================================================================
# Part 4: Temperature Response Functions (cf. Fig. 9)
# =====================================================================

def temperature_response(log_t, log_t_peak, sigma, amplitude, secondary=None):
    """Compute a simplified temperature response function.

    Models the response as a Gaussian in log T space, optionally with
    a secondary peak (for the 304 A channel which has He II + Si XI).

    Args:
        log_t: Array of log10(T) values.
        log_t_peak: Peak formation temperature (log10 K).
        sigma: Width of the response in log T.
        amplitude: Peak amplitude (DN/s/pixel per unit EM).
        secondary: Optional dict with 'log_t', 'sigma', 'amplitude' for
            a secondary peak.

    Returns:
        Temperature response array.
    """
    response = amplitude * np.exp(-0.5 * ((log_t - log_t_peak) / sigma)**2)
    if secondary is not None:
        response += secondary['amplitude'] * np.exp(
            -0.5 * ((log_t - secondary['log_t']) / secondary['sigma'])**2)
    return response


# Temperature grid.
log_t = np.linspace(4.0, 7.5, 500)

# Response parameters for each EIT band.
response_params = {
    '171': {'log_t_peak': 5.9, 'sigma': 0.25, 'amplitude': 2.5e-25},
    '195': {'log_t_peak': 6.2, 'sigma': 0.30, 'amplitude': 4.0e-25},
    '284': {'log_t_peak': 6.3, 'sigma': 0.20, 'amplitude': 1.5e-25},
    '304': {'log_t_peak': 4.9, 'sigma': 0.30, 'amplitude': 8.0e-25,
            'secondary': {'log_t': 6.15, 'sigma': 0.25, 'amplitude': 1.0e-25}},
}

# --- 4-panel plot (cf. Fig. 9) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('EIT Temperature Response Functions G(T) (cf. Fig. 9)',
             fontsize=14, fontweight='bold')

for ax, (band, params) in zip(axes.flat, response_params.items()):
    info = EIT_BANDS[band]
    sec = params.get('secondary', None)
    g_t = temperature_response(log_t, params['log_t_peak'], params['sigma'],
                                params['amplitude'], secondary=sec)

    ax.semilogy(log_t, g_t, color=info['color'], linewidth=2.5)
    ax.set_title(f"{band} A  ({info['ion']})", fontsize=12)
    ax.set_xlabel('log T (K)')
    ax.set_ylabel('G(T)  (DN cm^5 s^-1 pixel^-1)')
    ax.set_xlim(4.0, 7.5)
    ax.set_ylim(1e-28, 1e-23)
    ax.axvline(params['log_t_peak'], color='gray', linestyle=':', alpha=0.5)
    ax.text(params['log_t_peak'] + 0.05, params['amplitude'] * 0.5,
            f'log T = {params["log_t_peak"]:.1f}', fontsize=9, color='gray')
    if sec:
        ax.axvline(sec['log_t'], color='gray', linestyle=':', alpha=0.3)
        ax.text(sec['log_t'] + 0.05, sec['amplitude'] * 0.5,
                f'log T = {sec["log_t"]:.2f}\n(Si XI)', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

# --- Combined overlay plot ---
fig, ax = plt.subplots(figsize=(10, 6))

for band, params in response_params.items():
    info = EIT_BANDS[band]
    sec = params.get('secondary', None)
    g_t = temperature_response(log_t, params['log_t_peak'], params['sigma'],
                                params['amplitude'], secondary=sec)
    g_norm = g_t / g_t.max()
    ax.semilogy(log_t, g_norm, color=info['color'], linewidth=2.5,
                label=f"{band} A ({info['ion']})")

ax.set_xlabel('log T (K)', fontsize=12)
ax.set_ylabel('Normalized G(T)', fontsize=12)
ax.set_title('EIT Temperature Coverage: 10^4 - 10^7 K',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(4.0, 7.5)
ax.set_ylim(1e-4, 2.0)

# Annotate temperature regimes.
ax.axvspan(4.5, 5.3, alpha=0.05, color='red')
ax.axvspan(5.7, 6.5, alpha=0.05, color='blue')
ax.text(4.9, 1.2, 'Chromosphere\n/ Transition Region', ha='center',
        fontsize=9, style='italic')
ax.text(6.1, 1.2, 'Corona', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.show()

print("\nPeak formation temperatures:")
for band, params in response_params.items():
    t_peak = 10**params['log_t_peak']
    print(f"  {band} A: log T = {params['log_t_peak']:.1f}  "
          f"(T = {t_peak:.1e} K, {EIT_BANDS[band]['ion']})")

## Part 5: Data Compression Analysis / 데이터 압축 분석

EIT faces severe telemetry constraints: only ~1 kbit/s allocated bandwidth from the L1 Lagrange point. The paper describes multiple compression strategies (Table II, Section 6.3.6):

EIT는 심각한 텔레메트리 제약에 직면합니다: L1 라그랑주 점에서 할당된 대역폭이 ~1 kbit/s에 불과합니다. 논문은 여러 압축 전략을 설명합니다:

1. **Square root compression**: 14-bit -> 7-bit lossless (preserving photon noise statistics)
2. **ADCT**: Adaptive Discrete Cosine Transform for lossy compression (factors of 5-20)
3. **On-board pixel summing**: 2x2 or 4x4 binning to reduce data volume

These techniques are essential for achieving reasonable image cadence (~12-15 min).

이러한 기술들은 합리적인 영상 케이던스(~12-15분)를 달성하는 데 필수적입니다.

In [ ]:
# =====================================================================
# Part 5a: Square root compression (14-bit -> 7-bit)
# =====================================================================
dn_14bit = np.arange(0, 2**14)  # 0 to 16383

# Square root compression: compress 14-bit to 7-bit (128 levels).
scale = (2**7 - 1)**2 / (2**14 - 1)
dn_compressed = np.round(np.sqrt(dn_14bit * scale)).astype(int)
dn_compressed = np.clip(dn_compressed, 0, 127)

# Decompression (inverse).
dn_decompressed = (dn_compressed.astype(float))**2 / scale

# Quantization error.
quant_error = dn_14bit - dn_decompressed

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Compression mapping.
ax = axes[0]
ax.plot(dn_14bit, dn_compressed, 'b-', linewidth=0.5)
ax.set_xlabel('Original DN (14-bit)')
ax.set_ylabel('Compressed value (7-bit)')
ax.set_title('Square Root Compression Mapping')

# (b) Quantization noise vs signal.
ax = axes[1]
ax.scatter(dn_14bit[::10], np.abs(quant_error[::10]), s=1, alpha=0.3, color='red')
dn_range = np.linspace(1, 2**14, 500)
ax.plot(dn_range, np.sqrt(dn_range), 'k--', linewidth=2,
        label='Photon noise (sqrt(DN))')
ax.set_xlabel('Original DN')
ax.set_ylabel('Error (DN)')
ax.set_title('Quantization Error vs Signal')
ax.legend()
ax.set_xlim(0, 5000)
ax.set_ylim(0, 100)

# (c) Relative error.
ax = axes[2]
nonzero = dn_14bit > 10
rel_error = np.abs(quant_error[nonzero]) / dn_14bit[nonzero] * 100
ax.plot(dn_14bit[nonzero][::10], rel_error[::10], 'g-', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Original DN')
ax.set_ylabel('Relative error (%)')
ax.set_title('Relative Quantization Error')
ax.set_ylim(0, 10)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='1% level')
ax.legend()

plt.suptitle('Square Root Compression: 14-bit -> 7-bit (2:1 ratio)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"14-bit range: 0 - {2**14-1}")
print(f"7-bit range:  0 - {2**7-1}")
print(f"Compression ratio: {14/7:.1f}:1")

In [ ]:
# =====================================================================
# Part 5b: Telemetry budget and image cadence
# =====================================================================
TELEMETRY_RATE = 1000  # bits per second (1 kbit/s)

# Image sizes for different compression modes.
modes = {
    'Uncompressed\n(14-bit)': 1024**2 * 14,
    'Sqrt\n(7-bit)': 1024**2 * 7,
    'ADCT 10x': 1024**2 * 14 / 10,
    'ADCT 20x': 1024**2 * 14 / 20,
    '4x4 sum\n+ sqrt': 256**2 * 7,
    '4x4 sum\n+ ADCT 10x': 256**2 * 14 / 10,
}

# Compute transfer times.
print("=" * 65)
print("Telemetry Budget at 1 kbit/s")
print("=" * 65)
print(f"{'Mode':<22s} {'Size (Mbit)':>12s} {'Time':>15s}")
print("-" * 65)

mode_names = []
cadence_minutes = []

for mode, bits in modes.items():
    size_mbit = bits / 1e6
    time_sec = bits / TELEMETRY_RATE
    time_min = time_sec / 60.0
    time_hr = time_sec / 3600.0

    mode_clean = mode.replace('\n', ' ')
    if time_hr > 1:
        time_str = f"{time_hr:.1f} hours"
    else:
        time_str = f"{time_min:.1f} min"

    print(f"{mode_clean:<22s} {size_mbit:>12.2f} {time_str:>15s}")
    mode_names.append(mode)
    cadence_minutes.append(time_min)

# Bar chart of cadences.
fig, ax = plt.subplots(figsize=(12, 6))

colors_bar = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd', '#8c564b']
bars = ax.bar(range(len(modes)), cadence_minutes, color=colors_bar,
              edgecolor='black', linewidth=0.5)

ax.set_xticks(range(len(modes)))
ax.set_xticklabels(mode_names, fontsize=10)
ax.set_ylabel('Image cadence (minutes)', fontsize=12)
ax.set_title('EIT Image Cadence vs Compression Mode (at 1 kbit/s)',
             fontsize=13, fontweight='bold')

# Add value labels on bars.
for bar, val in zip(bars, cadence_minutes):
    if val > 60:
        label = f'{val/60:.1f} hr'
    else:
        label = f'{val:.0f} min'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, max(cadence_minutes) * 1.15)

# Reference line: typical EIT cadence.
ax.axhline(12, color='red', linestyle='--', alpha=0.7, linewidth=2)
ax.text(len(modes) - 0.5, 14, 'Typical EIT cadence (~12 min)',
        fontsize=10, color='red', ha='right')

plt.tight_layout()
plt.show()

print(f"\nAt 1 kbit/s, uncompressed 1024x1024x14-bit image takes "
      f"{1024**2*14/TELEMETRY_RATE/3600:.1f} hours!")
print("ADCT compression is essential for reasonable cadence.")

## Part 6: EIT vs AIA Comparison / EIT vs AIA 비교

EIT (launched 1995 on SOHO) was the pioneering full-disk EUV imager. Its direct successor, SDO/AIA (launched 2010), represents 15 years of technological progress. The improvements in every parameter are dramatic.

EIT(1995년 SOHO에 탑재 발사)는 선구적인 전체 디스크 EUV 영상 장치였습니다. 직접 후속 기기인 SDO/AIA(2010년 발사)는 15년간의 기술 발전을 대표합니다. 모든 파라미터에서의 개선은 극적입니다.

| Parameter | EIT (1995) | AIA (2010) | Improvement |
|-----------|-----------|-----------|-------------|
| Aperture | 12 cm | 20 cm | 1.7x |
| Focal length | 165 cm | 460 cm | 2.8x |
| Detector | 1024x1024 | 4096x4096 | 16x pixels |
| Pixel scale | 2.6" | 0.6" | 4.3x finer |
| FOV | 45' | 41' | comparable |
| Bands | 4 EUV | 7 EUV + 2 UV | 2.25x |
| Cadence | ~12 min | 12 s | 60x faster |
| Data rate | 1 kbit/s | ~67 Mbit/s | 67,000x |

In [ ]:
# =====================================================================
# Part 6: EIT vs AIA comparison bar chart
# =====================================================================
parameters = ['Aperture', 'Focal\nlength', 'Detector\npixels',
              'Spatial\nresolution', 'Spectral\nbands', 'Cadence',
              'Data rate']
eit_values = [12, 165, 1024**2, 2.62, 4, 12*60, 1e3]
aia_values = [20, 460, 4096**2, 0.6, 9, 12, 67e6]

# Compute improvement factors.
improvement = []
for i, (e, a) in enumerate(zip(eit_values, aia_values)):
    if parameters[i] in ['Spatial\nresolution', 'Cadence']:
        imp = e / a  # Lower is better.
    else:
        imp = a / e
    improvement.append(imp)

fig, ax = plt.subplots(figsize=(12, 6))

colors_imp = ['#2ca02c' if imp > 10 else '#1f77b4' if imp > 2
              else '#ff7f0e' for imp in improvement]
bars = ax.bar(range(len(parameters)), improvement, color=colors_imp,
              edgecolor='black', linewidth=0.5)

ax.set_xticks(range(len(parameters)))
ax.set_xticklabels(parameters, fontsize=11)
ax.set_ylabel('Improvement factor (AIA / EIT)', fontsize=12)
ax.set_title('SDO/AIA vs SOHO/EIT: 15 Years of Progress',
             fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.set_ylim(0.5, 2e5)

# Value labels.
for bar, imp in zip(bars, improvement):
    label = f'{imp:.0f}x' if imp >= 2 else f'{imp:.1f}x'
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.3,
            label, ha='center', va='bottom', fontsize=11, fontweight='bold')

# Highlight the most dramatic improvement.
ax.annotate('67,000x data rate!',
            xy=(6, improvement[6]), xytext=(4.5, improvement[6] * 2),
            fontsize=11, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

# Print comparison table.
print("\n" + "=" * 70)
print("EIT vs AIA Detailed Comparison")
print("=" * 70)
print(f"{'Parameter':<20s} {'EIT (1995)':>15s} {'AIA (2010)':>15s} {'Factor':>10s}")
print("-" * 70)
table_data = [
    ('Aperture', '12 cm', '20 cm', '1.7x'),
    ('Focal length', '165 cm', '460 cm', '2.8x'),
    ('Detector', '1024x1024', '4096x4096', '16x pixels'),
    ('Pixel scale', '2.6"', '0.6"', '4.3x finer'),
    ('FOV', "45'", "41'", '~1x'),
    ('EUV bands', '4', '7 + 2 UV', '2.25x'),
    ('Cadence', '~12 min', '12 s', '60x'),
    ('Data rate', '1 kbit/s', '67 Mbit/s', '67,000x'),
]
for param, eit, aia, factor in table_data:
    print(f"{param:<20s} {eit:>15s} {aia:>15s} {factor:>10s}")

## Summary / 요약

### EIT in the Arc of Solar EUV Imaging / 태양 EUV 영상의 역사 속 EIT

EIT was not the first EUV solar imager, but it was the first to provide **continuous, full-disk EUV monitoring** from space. Its 4-quadrant multilayer design was innovative and influential.

EIT는 최초의 EUV 태양 영상 장치는 아니었지만, 우주에서 **연속적인 전체 디스크 EUV 모니터링**을 최초로 제공했습니다. 4사분면 다층 코팅 설계는 혁신적이고 영향력이 있었습니다.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Multilayer mirrors / 다층 코팅 거울 | Mo/Si 4-quadrant, N=20-40, R~3-7% | Mo/Si single-band per telescope, R~30-40% |
| Detector / 검출기 | 1024x1024 CCD, 21 um pixel | 4096x4096 CCD/CMOS, 12 um pixel |
| Compression / 압축 | Sqrt + ADCT, 1 kbit/s | Rice/JPEG2000, 67 Mbit/s |
| Temperature diagnostics / 온도 진단 | 4-band filter ratio | DEM inversion with 6+ bands |
| Cadence / 케이던스 | ~12 min (best) | 12 s (AIA), sub-second (Hi-C) |

In [ ]:
# =====================================================================
# Summary: EIT in context — predecessor, contemporaries, successors
# =====================================================================
instruments = [
    {'name': 'Skylab ATM', 'year': 1973, 'aperture': '~3 cm',
     'detector': 'Film', 'pixel': '~2"', 'fov': "32'",
     'bands': '6 (171-335 A)', 'cadence': 'hours',
     'note': 'First solar EUV images from orbit'},
    {'name': 'SOHO/EIT', 'year': 1995, 'aperture': '12 cm',
     'detector': '1024x1024 CCD', 'pixel': '2.6"', 'fov': "45'",
     'bands': '4 (171-304 A)', 'cadence': '~12 min',
     'note': 'First continuous full-disk EUV'},
    {'name': 'TRACE', 'year': 1998, 'aperture': '30 cm',
     'detector': '1024x1024 CCD', 'pixel': '0.5"', 'fov': "8.5'",
     'bands': '3 EUV + UV', 'cadence': '~30 s',
     'note': 'High-res but small FOV'},
    {'name': 'STEREO/EUVI', 'year': 2006, 'aperture': '~10 cm',
     'detector': '2048x2048 CCD', 'pixel': '1.6"', 'fov': "54'",
     'bands': '4 (171-304 A)', 'cadence': '~5 min',
     'note': 'Dual viewpoint (stereo)'},
    {'name': 'SDO/AIA', 'year': 2010, 'aperture': '20 cm',
     'detector': '4096x4096 CCD', 'pixel': '0.6"', 'fov': "41'",
     'bands': '7 EUV + 2 UV', 'cadence': '12 s',
     'note': 'High cadence, high resolution'},
    {'name': 'SO/EUI-FSI', 'year': 2020, 'aperture': '~5 cm',
     'detector': '2048x2048 CMOS', 'pixel': '4.4"', 'fov': "228'",
     'bands': '2 (174, 304 A)', 'cadence': '~10 min',
     'note': 'Ultra-wide FOV, inner heliosphere'},
]

# Print comparison table.
print("=" * 110)
print("Solar EUV Full-Disk Imagers: Skylab to Solar Orbiter")
print("=" * 110)
header = (f"{'Instrument':<16s} {'Year':>5s} {'Aperture':>10s} {'Detector':>16s} "
          f"{'Pixel':>7s} {'FOV':>7s} {'Bands':>16s} {'Cadence':>10s}")
print(header)
print("-" * 110)

for inst in instruments:
    line = (f"{inst['name']:<16s} {inst['year']:>5d} {inst['aperture']:>10s} "
            f"{inst['detector']:>16s} {inst['pixel']:>7s} {inst['fov']:>7s} "
            f"{inst['bands']:>16s} {inst['cadence']:>10s}")
    print(line)
    print(f"{'':>16s} Note: {inst['note']}")

# Timeline visualization.
fig, ax = plt.subplots(figsize=(14, 4))

for i, inst in enumerate(instruments):
    color = '#d62728' if inst['name'] == 'SOHO/EIT' else '#1f77b4'
    marker_size = 15 if inst['name'] == 'SOHO/EIT' else 10
    ax.scatter(inst['year'], 0, s=marker_size**2, color=color, zorder=5)
    va = 'bottom' if i % 2 == 0 else 'top'
    y_offset = 0.3 if i % 2 == 0 else -0.3
    ax.annotate(f"{inst['name']}\n({inst['year']})",
                xy=(inst['year'], 0), xytext=(inst['year'], y_offset),
                ha='center', va='center', fontsize=9,
                fontweight='bold' if inst['name'] == 'SOHO/EIT' else 'normal',
                color=color,
                arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

ax.axhline(0, color='gray', linewidth=2, alpha=0.3)
ax.set_xlim(1970, 2025)
ax.set_ylim(-0.8, 0.8)
ax.set_xlabel('Year', fontsize=12)
ax.set_title('Timeline of Solar EUV Full-Disk Imagers',
             fontsize=13, fontweight='bold')
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n--- Key Legacy of EIT ---")
print("1. Demonstrated continuous full-disk EUV monitoring from L1")
print("2. Pioneered 4-quadrant multilayer mirror design")
print("3. Provided 15+ years of uninterrupted solar observations (1996-2011+)")
print("4. Enabled discovery of EIT waves, coronal dimming, and CME source regions")
print("5. Set the template for all subsequent full-disk EUV imagers")